# Exact parameter recovery for Shimura-curve CM expansions

**SageMath kernel required.** Choose **SageMath**, then **Restart Kernel and Run All**.
Alternatively, run the setup cells once and then run any one of the nine form sections.
All data and code are embedded in this notebook; no accompanying files, network access,
or numerical coefficient files are needed.

The forms covered are **33a, 34a, 46a, 26a, 26b, 38a, 38b, 58a, 58b**.
For each form the notebook displays:

1. The input $H(u)$, the affine family for $\mathcal R(u)$, the marked value $u_0$, and the assumed $c_n$.
2. The exact rational functions $L(u_1),C(u_1),B(u_1)$, or $L(u_1),C(u_1),B_0(u_1),B_1(u_1)$.
3. The coordinate derivatives $u_i=u^{(i)}(0)$ and the residual equations $E_j(u_1)=0$.
4. The unique common root $u_1$ of those equations, and every recovered parameter.
5. An exact check of every input coefficient and one new coefficient predicted by the recurrence.

The geometric input and the initial coefficient identities are **hypotheses**.
Exact algebra proves recovery and uniqueness from those hypotheses; it does not prove numerical recognitions.
The optional reference checks at the end are kept separate from the solver and are never used to find a root.


## Equations and conventions

We use exponential coefficients and the original tangent variable:
\[
A(q)=\sum_{n\ge0}c_n\frac{q^n}{n!},\quad c_0=c_1=1,
\qquad u(q)=\sum_{n\ge0}u_n\frac{q^n}{n!}.
\]
The equations are
\[
AA''-\frac32(A')^2+L\mathcal R(u)A^4=0,
\qquad u''=\frac{A'}A u'+\frac L2 A^2H'(u),
\qquad L=\frac{u_1^2}{H(u_0)}.
\]
Primes on series mean $d/dq$; primes on $H,\mathcal R$ mean $d/du$.
The computation takes place in $\mathbb Q(u_1)[[q]]$, with $u_1\ne0$.
It reconstructs $L\mathcal R(u(q))$ directly as
\[
\frac{\frac32(A')^2-AA''}{A^4}.
\]
For $d$ accessory constants, matching derivatives of orders $0,\ldots,d-1$
gives a linear system for those constants, with $u_1$ still symbolic.
Thus two accessories use $c_2,c_3$; three accessories use $c_2,c_3,c_4$.

Define the remaining **derivative** residuals (not ordinary series coefficients) by
\[
E_j(u_1)=\left.\frac{d^j}{dq^j}
\left(L\mathcal R(u(q))-\frac{\frac32(A')^2-AA''}{A^4}\right)\right|_{q=0}.
\]
The code multiplies the coefficient of $q^j$ by $j!$.
We then solve the remaining equations directly:

- Two accessory constants: $E_2(u_1)=E_3(u_1)=0$.
- Three accessory constants, form $58a$: $E_3(u_1)=E_4(u_1)=0$.
- Three accessory constants, form $58b$: $E_3(u_1)=0$ already has a unique root.

All these residuals are already polynomials in $u_1$ with rational coefficients.
Internally, their polynomial gcd identifies their common roots exactly.
The solver checks the excluded value $u_1=0$, the elimination denominators, and
uniqueness over $\mathbb C$. The displayed residual equations are followed immediately
by the recovered tangent and parameters.


In [1]:
from sage.all import (
    QQ, PolynomialRing, PowerSeriesRing, LaurentPolynomialRing,
    matrix, vector, factorial, latex
)
from IPython.display import display, Math, Markdown
from sage.version import version as sage_version

# Ordinary Python syntax is used throughout: ** for powers, exact QQ inputs.
Qu = PolynomialRing(QQ, 'u')
u = Qu.gen()
Fu = Qu.fraction_field()
Qt = PolynomialRing(QQ, 'u1')
u1 = Qt.gen()
K = Qt.fraction_field()
Laurent = LaurentPolynomialRing(QQ, 'u1')
lu1 = Laurent.gen()
cases, fits, results = {}, {}, {}
print('SageMath', sage_version)


SageMath 9.5


## Shared exact computation

The next two cells contain the engine and its display functions.
Each later form supplies only its family, $H$, $u_0$, and the assumed coefficients.
Series are rebuilt when a new coordinate coefficient is appended, so no coefficient
is lost through an earlier $O(q^k)$ truncation. Inputs through $c_m$ are used only
for residuals through $E_{m-2}$.


In [2]:
def make_case(label, H, Phi, poles, u0, c):
    """Only geometric/family data and exact initial identities enter here."""
    H, Phi, u0 = Qu(H), Fu(Phi), QQ(u0)
    poles, c = [QQ(a) for a in poles], [QQ(t) for t in c]
    names = ['C', 'B'] if len(poles) == 1 else ['C', 'B_0', 'B_1']
    if len(poles) not in (1, 2) or len(set(poles)) != len(poles):
        raise ValueError('Use one or two distinct accessory poles.')
    assert c[:2] == [1, 1] and len(c) >= len(names) + 2
    assert H(u0) != 0 and all(u0 != a for a in poles)
    assert Phi.denominator()(u0) != 0
    return dict(label=label, H=H, Phi=Phi, poles=poles, u0=u0, c=c,
                names=names, basis=[Fu(1)] + [1 / (Fu(u) - a) for a in poles])


def rational_on_series(f, U):
    """Evaluate a rational function of u on a q-series with regular center."""
    f = Fu(f)
    denominator = f.denominator()(U)
    if denominator[0] == 0:
        raise ValueError('The selected center is a pole.')
    return f.numerator()(U) / denominator


def monomial_polynomial(p):
    return bool(p) and len(p.exponents()) == 1


def only_zero_excluded(f):
    """A reduced denominator has no nonzero root precisely when it is monomial."""
    return monomial_polynomial(Qt(K(f).denominator()))


def evaluate_at_tangent(f, root):
    f = K(f)
    denominator = f.denominator()(root)
    if denominator == 0:
        raise ValueError('Candidate tangent lies in an excluded denominator locus.')
    return QQ(f.numerator()(root) / denominator)


def as_laurent(f):
    """Display a rational function as powers of u1, including negative powers."""
    f = K(f)
    denominator = Qt(f.denominator())
    if not monomial_polynomial(denominator):
        return f
    degree = denominator.exponents()[0]
    return sum((QQ(a) / denominator[degree]) * lu1**(i-degree)
               for i, a in enumerate(f.numerator().list()) if a)


def exclude_zero_root(p):
    """The geometric condition u1 != 0 is imposed before testing uniqueness."""
    p = Qt(p)
    while p and p[0] == 0:
        p, remainder = p.quo_rem(u1)
        assert remainder == 0
    return p.monic() if p else p


def eliminate_parameters(case):
    c, H, Phi, u0 = case['c'], case['H'], case['Phi'], case['u0']
    d, m = len(case['names']), len(c) - 1
    last_j = m - 2
    Qq = PowerSeriesRing(K, 'q', default_prec=m+4)
    q = Qq.gen()
    A = Qq([K(c[n] / factorial(n)) for n in range(m+1)]).add_bigoh(m+1)
    connection_from_A = ((QQ(3)/2) * A.derivative()**2
                         - A * A.derivative().derivative()) / A**4
    assert connection_from_A.prec() >= last_j + 1
    L = K(u1)**2 / H(u0)

    # At step n, the right side uses only coefficients already known.
    # Rebuild U as an exact polynomial representative before extracting q^n.
    u_coefficients = [K(u0), K(u1)]
    for n in range(last_j - 1):
        U = Qq(u_coefficients)
        rhs = (A.derivative()/A) * U.derivative() + (L/2)*A**2*H.derivative()(U)
        u_coefficients.append(K(rhs[n]) / ((n+1)*(n+2)))
    U = Qq(u_coefficients).add_bigoh(last_j+1)
    fixed = rational_on_series(Phi, U)
    basis_series = [rational_on_series(f, U) for f in case['basis']]
    M = matrix(K, [[factorial(i)*L*basis_series[j][i] for j in range(d)]
                   for i in range(d)])
    target = vector(K, [factorial(i)*(connection_from_A[i]-L*fixed[i])
                        for i in range(d)])
    determinant = M.det()
    assert determinant != 0
    # Thus no singular nonzero tangent was lost in linear elimination.
    assert monomial_polynomial(Qt(determinant.numerator()))
    assert monomial_polynomial(Qt(determinant.denominator()))
    accessories = M.solve_right(target)
    parameters = {'L': L}
    parameters.update(zip(case['names'], accessories))
    assert all(only_zero_excluded(value) for value in parameters.values())
    fitted_connection = L*(fixed + sum(accessories[j]*basis_series[j] for j in range(d)))
    residuals = {j: K(factorial(j)*(fitted_connection[j]-connection_from_A[j]))
                 for j in range(last_j+1)}
    assert all(residuals[j] == 0 for j in range(d))
    # For these families the unmodified derivative residuals are already
    # polynomials over QQ. No rescaling or residual combinations are needed.
    assert all(K(value).denominator().degree() == 0 for value in residuals.values())
    equations = {j: Qt(residuals[j]) for j in range(d, last_j+1)}
    return dict(case=case, m=m, d=d, U=U, u_coefficients=u_coefficients,
                parameters=parameters, matrix=M, target=target, determinant=determinant,
                connection_from_A=connection_from_A, residuals=residuals,
                equations=equations)



def generate_coefficients(case, solution, last_index):
    """Coupled recurrence, using recovered parameters only (no further seed values)."""
    Qq = PowerSeriesRing(QQ, 'q', default_prec=last_index+4)
    H, u0, L = case['H'], case['u0'], solution['L']
    R = case['Phi'] + sum(solution[name]*basis
                          for name, basis in zip(case['names'], case['basis']))
    a, b = [QQ(1), QQ(1)], [u0, solution['u1']]
    for n in range(last_index-1):
        A, U = Qq(a), Qq(b)
        Aprime = A.derivative()
        rhs_A = (QQ(3)/2)*Aprime**2/A - L*rational_on_series(R,U)*A**3
        rhs_u = (Aprime/A)*U.derivative() + (L/2)*A**2*H.derivative()(U)
        denominator = (n+1)*(n+2)
        next_a, next_b = QQ(rhs_A[n])/denominator, QQ(rhs_u[n])/denominator
        a.append(next_a)
        b.append(next_b)
    A, U = Qq(a), Qq(b)
    first_integral = U.derivative()**2 - L*A**2*H(U)
    assert all(first_integral[j] == 0 for j in range(last_index))
    return [QQ(a[n]*factorial(n)) for n in range(last_index+1)], Fu(R)


def recover_parameters(fit):
    # Solve E_j(u1)=0 simultaneously: their gcd has exactly the common roots.
    common = Qt(0)
    for equation in fit['equations'].values():
        common = common.gcd(equation)
    common = exclude_zero_root(common)
    recovered = dict(fit=fit, gcd=common, solution=None)
    if common.degree() != 1:
        return recovered
    root = QQ(-common[0]/common[1])
    assert root != 0
    assert evaluate_at_tangent(fit['determinant'], root) != 0
    solution = {'u1': root}
    solution.update({name: evaluate_at_tangent(f, root)
                     for name, f in fit['parameters'].items()})
    assert all(evaluate_at_tangent(f, root) == 0 for f in fit['residuals'].values())
    assert root**2 == solution['L']*fit['case']['H'](fit['case']['u0'])
    last_index = fit['m'] + 1
    coefficients, R = generate_coefficients(fit['case'], solution, last_index)
    assert coefficients[:len(fit['case']['c'])] == fit['case']['c']
    recovered.update(solution=solution, R=R, coefficients=coefficients,
                     predicted_index=last_index, predicted_value=coefficients[last_index])
    return recovered


In [3]:
def math_line(label, value):
    display(Math(label + ' = ' + latex(value)))


def show_case(case):
    math_line(r'H(u)', case['H'])
    family = 'C'
    for name, pole in zip(case['names'][1:], case['poles']):
        family += '+' + name + r'\left(' + latex(1/(Fu(u)-pole)) + r'\right)'
    family += r'+\left(' + latex(case['Phi']) + r'\right)'
    display(Math(r'\mathcal R(u)=' + family))
    math_line('u_0', case['u0'])
    math_line(r'H(u_0)', case['H'](case['u0']))
    for n in range(2, len(case['c'])):
        math_line('c_{%s}' % n, case['c'][n])


def show_elimination(fit):
    print('Matching derivative orders 0 through %s eliminates %s.' %
          (fit['d']-1, ', '.join(fit['case']['names'])))
    math_line(r'\det M', as_laurent(fit['determinant']))
    print('The determinant is nonzero for every u1 != 0; no nonzero singular branch is omitted.')
    for name, value in fit['parameters'].items():
        math_line(name + '(u_1)', as_laurent(value))
    print('Coordinate derivatives needed for the remaining constraints:')
    for n in range(2, len(fit['u_coefficients'])):
        math_line('u_{%s}' % n, as_laurent(factorial(n)*fit['u_coefficients'][n]))


def show_recovery(result):
    fit = result['fit']
    print('Solving the residual equations:')
    for j, equation in sorted(fit['equations'].items()):
        display(Math('E_{%s}(u_1) = %s = 0' % (j, latex(as_laurent(equation)))))
    if result['solution'] is None:
        if result['gcd'] == 0:
            print('These residuals impose no constraint on the nonzero tangent.')
        elif result['gcd'].degree() == 0:
            print('These residual equations have no common admissible solution.')
        else:
            print('A unique tangent has not been established: %s distinct common complex roots remain.' %
                  result['gcd'].squarefree_part().degree())
        return
    print('Unique common solution on the admissible domain:')
    for name, value in result['solution'].items():
        math_line(name, value)
    math_line(r'\mathcal R(u)', result['R'])
    print('PASS: all assumed coefficients, original residual equations, denominator exclusions, and first integral.')
    print('First coefficient beyond the input range, computed from the recovered recurrence:')
    math_line('c_{%s}' % result['predicted_index'], result['predicted_value'])


## 33a

Two accessory constants. Inputs through $c_5$ give $E_2(u_1)=E_3(u_1)=0$; their common root determines the tangent.

Run these three cells after the shared setup. Edit only the input cell to experiment.


In [4]:
cases["33a"] = make_case(
    "33a",
    H=4*u*(-3*u**2 - 10*u - 243),
    Phi=-384/(u-1)**2 - 108/u,
    poles=[QQ("1")],
    u0=QQ("9/25"),
    c=[
        QQ("1"),
        QQ("1"),
        QQ("-72337/529"),
        QQ("9202302/12167"),
        QQ("3009372468/279841"),
        QQ("-1149714703920/6436343")
    ],
)
show_case(cases["33a"])


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [5]:
fits["33a"] = eliminate_parameters(cases["33a"])
show_elimination(fits["33a"])


Matching derivative orders 0 through 1 eliminates C, B.


<IPython.core.display.Math object>

The determinant is nonzero for every u1 != 0; no nonzero singular branch is omitted.


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Coordinate derivatives needed for the remaining constraints:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [6]:
results["33a"] = recover_parameters(fits["33a"])
show_recovery(results["33a"])


Solving the residual equations:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

Unique common solution on the admissible domain:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: all assumed coefficients, original residual equations, denominator exclusions, and first integral.
First coefficient beyond the input range, computed from the recovered recurrence:


<IPython.core.display.Math object>

## 34a

Two accessory constants. Inputs through $c_5$ give $E_2(u_1)=E_3(u_1)=0$; their common root determines the tangent.

Run these three cells after the shared setup. Edit only the input cell to experiment.


In [7]:
cases["34a"] = make_case(
    "34a",
    H=(u**2 + 4)*(-3*u**2 + 26*u - 59),
    Phi=-QQ(1336336)/(16875*(u-QQ(16)/15)**2),
    poles=[QQ("16/15")],
    u0=QQ("-15/4"),
    c=[
        QQ("1"),
        QQ("1"),
        QQ("-12/25"),
        QQ("2762/125"),
        QQ("-17736/625"),
        QQ("-67648/125")
    ],
)
show_case(cases["34a"])


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [8]:
fits["34a"] = eliminate_parameters(cases["34a"])
show_elimination(fits["34a"])


Matching derivative orders 0 through 1 eliminates C, B.


<IPython.core.display.Math object>

The determinant is nonzero for every u1 != 0; no nonzero singular branch is omitted.


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Coordinate derivatives needed for the remaining constraints:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [9]:
results["34a"] = recover_parameters(fits["34a"])
show_recovery(results["34a"])


Solving the residual equations:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

Unique common solution on the admissible domain:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: all assumed coefficients, original residual equations, denominator exclusions, and first integral.
First coefficient beyond the input range, computed from the recovered recurrence:


<IPython.core.display.Math object>

## 46a

Two accessory constants. Inputs through $c_5$ give $E_2(u_1)=E_3(u_1)=0$; their common root determines the tangent.

Run these three cells after the shared setup. Edit only the input cell to experiment.


In [10]:
cases["46a"] = make_case(
    "46a",
    H=4*u*(-u**2 + 45*u - 512),
    Phi=-196608/(u-64)**2 - QQ(3)*u/8,
    poles=[QQ("64")],
    u0=QQ("3136/81"),
    c=[
        QQ("1"),
        QQ("1"),
        QQ("27824049/123402050"),
        QQ("-114816966372/484661551375"),
        QQ("-784666539922896/3807016486050625"),
        QQ("-3694325348311426944/5980822899585531875")
    ],
)
show_case(cases["46a"])


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [11]:
fits["46a"] = eliminate_parameters(cases["46a"])
show_elimination(fits["46a"])


Matching derivative orders 0 through 1 eliminates C, B.


<IPython.core.display.Math object>

The determinant is nonzero for every u1 != 0; no nonzero singular branch is omitted.


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Coordinate derivatives needed for the remaining constraints:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [12]:
results["46a"] = recover_parameters(fits["46a"])
show_recovery(results["46a"])


Solving the residual equations:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

Unique common solution on the admissible domain:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: all assumed coefficients, original residual equations, denominator exclusions, and first integral.
First coefficient beyond the input range, computed from the recovered recurrence:


<IPython.core.display.Math object>

## 26a

Two accessory constants. Inputs through $c_5$ give $E_2(u_1)=E_3(u_1)=0$; their common root determines the tangent.

Run these three cells after the shared setup. Edit only the input cell to experiment.


In [13]:
cases["26a"] = make_case(
    "26a",
    H=4*(-2*u**3 + 19*u**2 - 24*u - 169),
    Phi=-QQ(507)/(2*u**2),
    poles=[QQ("0")],
    u0=QQ("1"),
    c=[
        QQ("1"),
        QQ("1"),
        QQ("-16/147"),
        QQ("274/3087"),
        QQ("-2488/9261"),
        QQ("352768/1361367")
    ],
)
show_case(cases["26a"])


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [14]:
fits["26a"] = eliminate_parameters(cases["26a"])
show_elimination(fits["26a"])


Matching derivative orders 0 through 1 eliminates C, B.


<IPython.core.display.Math object>

The determinant is nonzero for every u1 != 0; no nonzero singular branch is omitted.


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Coordinate derivatives needed for the remaining constraints:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [15]:
results["26a"] = recover_parameters(fits["26a"])
show_recovery(results["26a"])


Solving the residual equations:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

Unique common solution on the admissible domain:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: all assumed coefficients, original residual equations, denominator exclusions, and first integral.
First coefficient beyond the input range, computed from the recovered recurrence:


<IPython.core.display.Math object>

## 26b

Two accessory constants. Inputs through $c_5$ give $E_2(u_1)=E_3(u_1)=0$; their common root determines the tangent.

Run these three cells after the shared setup. Edit only the input cell to experiment.


In [16]:
cases["26b"] = make_case(
    "26b",
    H=4*(-169*u**3 - 24*u**2 + 19*u - 2),
    Phi=-3/u**2,
    poles=[QQ("0")],
    u0=QQ("1"),
    c=[
        QQ("1"),
        QQ("1"),
        QQ("40"),
        QQ("-734"),
        QQ("-11240"),
        QQ("1308544")
    ],
)
show_case(cases["26b"])


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [17]:
fits["26b"] = eliminate_parameters(cases["26b"])
show_elimination(fits["26b"])


Matching derivative orders 0 through 1 eliminates C, B.


<IPython.core.display.Math object>

The determinant is nonzero for every u1 != 0; no nonzero singular branch is omitted.


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Coordinate derivatives needed for the remaining constraints:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [18]:
results["26b"] = recover_parameters(fits["26b"])
show_recovery(results["26b"])


Solving the residual equations:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

Unique common solution on the admissible domain:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: all assumed coefficients, original residual equations, denominator exclusions, and first integral.
First coefficient beyond the input range, computed from the recovered recurrence:


<IPython.core.display.Math object>

## 38a

Two accessory constants. Inputs through $c_5$ give $E_2(u_1)=E_3(u_1)=0$; their common root determines the tangent.

Run these three cells after the shared setup. Edit only the input cell to experiment.


In [19]:
cases["38a"] = make_case(
    "38a",
    H=4*(-16*u**3 - 59*u**2 - 82*u - 19),
    Phi=-QQ(57)/(2*u**2) - 6*u,
    poles=[QQ("0")],
    u0=QQ("1"),
    c=[
        QQ("1"),
        QQ("1"),
        QQ("-64/27"),
        QQ("-62/27"),
        QQ("-19672/2187"),
        QQ("41984/6561")
    ],
)
show_case(cases["38a"])


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [20]:
fits["38a"] = eliminate_parameters(cases["38a"])
show_elimination(fits["38a"])


Matching derivative orders 0 through 1 eliminates C, B.


<IPython.core.display.Math object>

The determinant is nonzero for every u1 != 0; no nonzero singular branch is omitted.


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Coordinate derivatives needed for the remaining constraints:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [21]:
results["38a"] = recover_parameters(fits["38a"])
show_recovery(results["38a"])


Solving the residual equations:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

Unique common solution on the admissible domain:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: all assumed coefficients, original residual equations, denominator exclusions, and first integral.
First coefficient beyond the input range, computed from the recovered recurrence:


<IPython.core.display.Math object>

## 38b

Two accessory constants. Inputs through $c_5$ give $E_2(u_1)=E_3(u_1)=0$; their common root determines the tangent.

Run these three cells after the shared setup. Edit only the input cell to experiment.


In [22]:
cases["38b"] = make_case(
    "38b",
    H=4*(-19*u**3 - 82*u**2 - 59*u - 16),
    Phi=-30/u**2,
    poles=[QQ("0")],
    u0=QQ("1"),
    c=[
        QQ("1"),
        QQ("1"),
        QQ("-16/169"),
        QQ("-878/2197"),
        QQ("7688/28561"),
        QQ("-2699648/371293")
    ],
)
show_case(cases["38b"])


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [23]:
fits["38b"] = eliminate_parameters(cases["38b"])
show_elimination(fits["38b"])


Matching derivative orders 0 through 1 eliminates C, B.


<IPython.core.display.Math object>

The determinant is nonzero for every u1 != 0; no nonzero singular branch is omitted.


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Coordinate derivatives needed for the remaining constraints:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [24]:
results["38b"] = recover_parameters(fits["38b"])
show_recovery(results["38b"])


Solving the residual equations:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

Unique common solution on the admissible domain:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: all assumed coefficients, original residual equations, denominator exclusions, and first integral.
First coefficient beyond the input range, computed from the recovered recurrence:


<IPython.core.display.Math object>

## 58a

Three accessory constants. Inputs through $c_6$ are used: $E_3=0$ alone has five candidates, and $E_4=0$ selects one.

Run these three cells after the shared setup. Edit only the input cell to experiment.


In [25]:
cases["58a"] = make_case(
    "58a",
    H=4*(-2*u**3 - 78*u**2 - 862*u - 1682),
    Phi=-2523/u**2 + QQ(65536)/(9*(u+27)**2),
    poles=[QQ("0"), QQ("-27")],
    u0=QQ("-11"),
    c=[
        QQ("1"),
        QQ("1"),
        QQ("-4/5"),
        QQ("314/125"),
        QQ("31192/625"),
        QQ("-1977728/625"),
        QQ("4428752/3125")
    ],
)
show_case(cases["58a"])


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [26]:
fits["58a"] = eliminate_parameters(cases["58a"])
show_elimination(fits["58a"])


Matching derivative orders 0 through 2 eliminates C, B_0, B_1.


<IPython.core.display.Math object>

The determinant is nonzero for every u1 != 0; no nonzero singular branch is omitted.


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Coordinate derivatives needed for the remaining constraints:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [27]:
results["58a"] = recover_parameters(fits["58a"])
show_recovery(results["58a"])


Solving the residual equations:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

Unique common solution on the admissible domain:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

... (2 outputs left out)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: all assumed coefficients, original residual equations, denominator exclusions, and first integral.
First coefficient beyond the input range, computed from the recovered recurrence:


<IPython.core.display.Math object>

## 58b

Three accessory constants. Inputs through $c_5$ suffice because $E_3(u_1)=0$ is linear.

Run these three cells after the shared setup. Edit only the input cell to experiment.


In [28]:
cases["58b"] = make_case(
    "58b",
    H=4*(-1682*u**3 - 862*u**2 - 78*u - 2),
    Phi=-3/u**2 - QQ(65536)/(177147*(u+QQ(1)/27)**2),
    poles=[QQ("0"), QQ("-1/27")],
    u0=QQ("-1/11"),
    c=[
        QQ("1"),
        QQ("1"),
        QQ("140/9"),
        QQ("262/3"),
        QQ("-39592/81"),
        QQ("420224/81")
    ],
)
show_case(cases["58b"])


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [29]:
fits["58b"] = eliminate_parameters(cases["58b"])
show_elimination(fits["58b"])


Matching derivative orders 0 through 2 eliminates C, B_0, B_1.


<IPython.core.display.Math object>

The determinant is nonzero for every u1 != 0; no nonzero singular branch is omitted.


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Coordinate derivatives needed for the remaining constraints:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [30]:
results["58b"] = recover_parameters(fits["58b"])
show_recovery(results["58b"])


Solving the residual equations:


<IPython.core.display.Math object>

Unique common solution on the admissible domain:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: all assumed coefficients, original residual equations, denominator exclusions, and first integral.
First coefficient beyond the input range, computed from the recovered recurrence:


<IPython.core.display.Math object>

## Why the two level-58 fits stop at different coefficients

This cell checks the special cancellations for $58b$ directly from the inputs.
The $c_2,c_3$ identity gives $\mathcal R'(u_0)=0$.
At the marked center, the fixed polar part and the accessory basis satisfy
\[
\mathcal R'''(u_0)=\frac{1419}{16}\mathcal R''(u_0)
-\frac{9801}{8}\mathcal R'(u_0).
\]
Together with the coordinate equation, this makes
$E_3=(85184/81)(33u_1+16)$ linear.
For $58a$, the equation $E_3=0$ has five admissible candidates; the additional equation $E_4=0$ from $c_6$ selects one.
These statements concern independent recovery of each form, with its center fixed.


In [31]:
case = cases['58b']
center, pole = case['u0'], case['poles'][1]
coefficient_R2 = -3*(2*center-pole)/(center*(center-pole))
coefficient_R1 = -6/(center*(center-pole))
for f in case['basis'] + [case['Phi']]:
    identity = f.derivative().derivative().derivative() - coefficient_R2*f.derivative().derivative() - coefficient_R1*f.derivative()
    assert identity(center) == 0
assert 6*case['c'][2] - case['c'][3] - 6 == 0
assert fits['58b']['residuals'][3] == K(QQ(85184)/81*(33*u1+16))
math_line(r'E_3^{58b}', as_laurent(fits['58b']['residuals'][3]))
assert fits['58b']['equations'][3].degree() == 1
assert fits['58a']['equations'][3].degree() == 5
assert fits['58a']['equations'][3].squarefree_part().degree() == 5
assert fits['58a']['equations'][3][0] != 0
assert results['58a']['gcd'].degree() == 1
print('PASS: 58b has a linear first constraint; 58a has five nonzero candidates before the c6 identity.')


<IPython.core.display.Math object>

PASS: 58b has a linear first constraint; 58a has five nonzero candidates before the c6 identity.


## Recovered parameters, collected

This summary uses only the results computed above. A dash means that the family has one accessory pole and its coefficient is named $B$.


In [32]:
rows = ['| Form | u1 | L | C | B or B0 | B1 |',
        '|:--|--:|--:|--:|--:|--:|']
for label, result in results.items():
    p = result['solution']
    assert p is not None, label + ': uniqueness not established'
    rows.append('| ' + ' | '.join([label, str(p['u1']), str(p['L']), str(p['C']),
                                  str(p.get('B', p.get('B_0'))), str(p.get('B_1', '—'))]) + ' |')
display(Markdown('\n'.join(rows)))


| Form | u1 | L | C | B or B0 | B1 |
|:--|--:|--:|--:|--:|--:|
| 33a | -19296/2875 | -67/529 | -4 | -96 | — |
| 34a | -3179/80 | -11/25 | 31/50 | 9248/1125 | — |
| 46a | -15188992/636255 | -844992/61701025 | -109/2 | -7168 | — |
| 26a | 44/21 | -11/1764 | 9/2 | -9 | — |
| 26b | 44 | -11/4 | 3 | 14 | — |
| 38a | 44/9 | -11/324 | -45/2 | -57 | — |
| 38b | 44/13 | -11/676 | -16 | -52 | — |
| 58a | -176/5 | 121/400 | 5 | -144358/243 | -229376/243 |
| 58b | -16/33 | -11/144 | -39625/243 | -34 | -163840/6561 |

## Optional comparison with the draft's recorded answers

The next cell is a regression check, **not an input to the recovery**.
The solver above has already determined every parameter and the first unused coefficient.
If you change a family or seed value to explore a different problem, skip or update this reference-check cell.


In [33]:
reference_answers = {
    "33a": {
        "parameters": {
            "u1": "-19296/2875",
            "L": "-67/529",
            "C": "-4",
            "B": "-96"
        },
        "prediction": {
            "index": 6,
            "value": "176790432531276/148035889"
        }
    },
    "34a": {
        "parameters": {
            "u1": "-3179/80",
            "L": "-11/25",
            "C": "31/50",
            "B": "9248/1125"
        },
        "prediction": {
            "index": 6,
            "value": "6553872/15625"
        }
    },
    "46a": {
        "parameters": {
            "u1": "-15188992/636255",
            "L": "-844992/61701025",
            "C": "-109/2",
            "B": "-7168"
        },
        "prediction": {
            "index": 6,
            "value": "939622156860406531404672/234896819381221764390625"
        }
    },
    "26a": {
        "parameters": {
            "u1": "44/21",
            "L": "-11/1764",
            "C": "9/2",
            "B": "-9"
        },
        "prediction": {
            "index": 6,
            "value": "5961776/28588707"
        }
    },
    "26b": {
        "parameters": {
            "u1": "44",
            "L": "-11/4",
            "C": "3",
            "B": "14"
        },
        "prediction": {
            "index": 6,
            "value": "5134480"
        }
    },
    "38a": {
        "parameters": {
            "u1": "44/9",
            "L": "-11/324",
            "C": "-45/2",
            "B": "-57"
        },
        "prediction": {
            "index": 6,
            "value": "-38071696/177147"
        }
    },
    "38b": {
        "parameters": {
            "u1": "44/13",
            "L": "-11/676",
            "C": "-16",
            "B": "-52"
        },
        "prediction": {
            "index": 6,
            "value": "25568336/4826809"
        }
    },
    "58a": {
        "parameters": {
            "u1": "-176/5",
            "L": "121/400",
            "C": "5",
            "B_0": "-144358/243",
            "B_1": "-229376/243"
        },
        "prediction": {
            "index": 7,
            "value": "-4829523664/78125"
        }
    },
    "58b": {
        "parameters": {
            "u1": "-16/33",
            "L": "-11/144",
            "C": "-39625/243",
            "B_0": "-34",
            "B_1": "-163840/6561"
        },
        "prediction": {
            "index": 6,
            "value": "-303950320/729"
        }
    }
}

for label, reference in reference_answers.items():
    computed = results[label]
    for name, value in reference['parameters'].items():
        assert computed['solution'][name] == QQ(value), (label, name)
    assert computed['predicted_index'] == reference['prediction']['index']
    assert computed['predicted_value'] == QQ(reference['prediction']['value'])
print('PASS: all nine recovered parameter tuples and all nine first predictions match the recorded answers.')


PASS: all nine recovered parameter tuples and all nine first predictions match the recorded answers.


The calculation uses SageMath's exact rational arithmetic, polynomial gcds, and power series.
Relevant implementation references: [power-series rings](https://doc.sagemath.org/html/en/reference/power_series/sage/rings/power_series_ring.html),
[power-series methods](https://doc.sagemath.org/html/en/reference/power_series/sage/rings/power_series_ring_element.html),
and [univariate polynomials](https://doc.sagemath.org/html/en/reference/polynomial_rings/sage/rings/polynomial/polynomial_element.html).
